# Crypto popularity -- distributions

Four plots, all treemaps or heatmaps -- size/color show two metrics at once
so each chart answers a "popularity by what?" question on its own, without
needing a table to read alongside it.

Period: **2026-01-01 to 2026-07-01** (same containment rule as the
dashboard: a market counts only if its whole lifetime fits inside the
period). Top 10 coins by volume, same as the dashboard's default.

In [1]:
import sqlite3
import pandas as pd
import plotly.express as px

DB_PATH = "../data/processed/markets.db"
PERIOD_START = "2026-01-01"
PERIOD_END_EXCLUSIVE = "2026-07-02"
TOP_N = 10

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("""
    SELECT m.id, c.name AS coin_name, m.start_date, s.volume, s.liquidity
    FROM markets m
    JOIN coins c ON c.id = m.coin_id
    JOIN market_snapshots s ON s.market_id = m.id
    WHERE m.start_date >= ? AND m.end_date < ?
""", conn, params=[PERIOD_START, PERIOD_END_EXCLUSIVE])
conn.close()

df["volume"] = df["volume"].fillna(0)
df["liquidity"] = df["liquidity"].fillna(0)
df["start_date"] = pd.to_datetime(df["start_date"], format="ISO8601", utc=True).dt.tz_localize(None)

by_coin = df.groupby("coin_name").agg(volume=("volume", "sum"), liquidity=("liquidity", "sum"), markets=("id", "count"))
by_coin["volume_per_market"] = by_coin["volume"] / by_coin["markets"]
top_coins = by_coin["volume"].sort_values(ascending=False).head(TOP_N).index
by_coin = by_coin.loc[top_coins]

df = df[df["coin_name"].isin(top_coins)]
print(f"{len(df):,} markets across the top {TOP_N} coins")
by_coin


498,549 markets across the top 10 coins


,volume,liquidity,markets,volume_per_market
coin_name,,,,
Bitcoin,7.342542e+09,13714.94960,109283,67188.325117
Ethereum,1.270606e+09,129205.16218,103651,12258.504231
Solana,4.360238e+08,1393.43802,74135,5881.483799
XRP,2.941004e+08,29704.84321,73852,3982.294809
Dogecoin,2.481289e+07,3612.39181,45747,542.393922
BNB,2.093753e+07,2566.93790,45871,456.443632
Hyperliquid,1.596005e+07,1252.23487,45884,347.834788
Chainlink,6.728142e+05,0.00000,66,10194.154237
Ethena,3.581454e+05,0.00000,59,6070.261612


## 1. Treemap -- size: volume, color: liquidity

Which coins dominate by money traded (size), and is that volume backed by a
deep order book or a thin one (color)? A coin that's big but pale is getting
traded without much resting depth behind it.

In [2]:
fig = px.treemap(
    by_coin.reset_index(), path=["coin_name"], values="volume", color="liquidity",
    color_continuous_scale="Blues",
    title="Volume (size) x Liquidity (color)",
)
fig.show()


## 2. Treemap -- size: market count, color: volume per market

Which coins have the most markets (size) -- and, separately, when a market
about that coin exists, how much does it actually get traded (color)? This
is the chart that exposes recurring-template inflation directly: a coin can
be huge on size (thousands of auto-generated 5-minute markets) and still be
pale on color if each of those markets barely trades.

In [3]:
fig = px.treemap(
    by_coin.reset_index(), path=["coin_name"], values="markets", color="volume_per_market",
    color_continuous_scale="Blues",
    title="Market count (size) x Volume per market (color)",
)
fig.show()


## 3. Heatmap -- volume by coin x month

The straightforward time distribution of the primary popularity metric:
which coins were carrying volume, and in which months.

In [4]:
df["month"] = df["start_date"].dt.to_period("M").dt.start_time
monthly_volume = df.groupby(["coin_name", "month"])["volume"].sum().reset_index()
pivot = monthly_volume.pivot(index="coin_name", columns="month", values="volume").reindex(top_coins).fillna(0)

fig = px.imshow(
    pivot, aspect="auto", color_continuous_scale=["#2a78d6", "#e34948"],
    labels=dict(x="Month", y="Coin", color="Volume"),
    title="Volume by coin x month (blue = low, red = high)",
)
fig.show()


## 4. Heatmap -- volume per market by coin x month

Same layout as #3, normalized by market count -- separates "this coin had a
big month" from "this coin had a lot of markets that month." A coin that's
hot on chart 3 but flat here was busy mostly because of market *count*, not
per-market engagement.

In [5]:
monthly = df.groupby(["coin_name", "month"]).agg(volume=("volume", "sum"), markets=("id", "count")).reset_index()
monthly["volume_per_market"] = monthly["volume"] / monthly["markets"]
pivot2 = monthly.pivot(index="coin_name", columns="month", values="volume_per_market").reindex(top_coins).fillna(0)

fig = px.imshow(
    pivot2, aspect="auto", color_continuous_scale=["#2a78d6", "#e34948"],
    labels=dict(x="Month", y="Coin", color="Volume / market"),
    title="Volume per market by coin x month (blue = low, red = high)",
)
fig.show()


## Takeaway

Size = "how big," color = "how good" -- each chart pairs a raw magnitude
with a normalized/secondary metric so both show up in one picture:
volume vs. liquidity depth (#1), market count vs. per-market engagement
(#2), and the same volume/engagement split again over time (#3 vs. #4).